# Mean-difference probe (per model organism)

A single submission notebook that, for **each** dataset the runner hands it, loads the
**model-organism-specific** probe trained in `tutorials/3.1-mean-probe.ipynb` and scores
with it.

Steps:

1. Identify the organism: get row 0 of `DATASET_NAME` for its `model` + `lora`
   columns; then pick the matching probe: each organism's `.npz` stores its own `model_id` /
   `lora_id`; we select the file whose pair matches this dataset (falling back to the
   base-model probe if a LoRA wasn't trained on).
2. Build that organism's model: `build_model(model, lora)` attaches the LoRA, so the
   eval activations come from the same adapter the probe was trained on.
3. Score: read the probe's layer, mean-pool the response span, project, calibrate
   (`bias` / `score_std`) -> sigmoid -> `[0,1]`, then `tracer.stop()` to skip later layers.

In [ ]:
import os

# ── DO NOT CHANGE ───────────────────────────────────────────────────
# The runner sets these per run; read them, don't hard-code or override.
#   DATASET_NAME : the eval dataset to predict on (encodes the organism).
#   NDIF_HOST / NDIF_API_KEY / HF_TOKEN : injected; nnsight + HF read them.
DATASET_NAME = os.environ["DATASET_NAME"]

# Optional `python submit.py --limit N` -> $ALETHEIA_LIMIT: score only the first N rows
# (fast rehearsal). Unset on the leaderboard -> every row.
LIMIT = int(os.environ["ALETHEIA_LIMIT"]) if os.environ.get("ALETHEIA_LIMIT") else None

## 1. Select this organism's probe

`tutorials/3.1-mean-probe.ipynb` saved one `mean_probe_*.npz` per organism, each tagged with
the `model_id` / `lora_id` it was trained on (plus `direction`, `layer`, and the calibration
`bias` / `score_std`). We read the organism off `DATASET_NAME`'s row 0 and pick the file whose
`(model_id, lora_id)` matches — falling back to the base-model probe (same model, no LoRA) if
this exact LoRA wasn't trained on.

In [ ]:
import sys, glob
import numpy as np

sys.path.insert(0, "submission")   # util.py lives in submission/; the runner starts at repo root
import util

# The organism is defined by the dataset's own columns (NOT the dataset name): peek row 0.
ex0 = util.load_examples(DATASET_NAME)[0]
DS_MODEL = ex0["model"]
DS_LORA  = ex0.get("lora")                       # None for a plain base organism

def _norm(v):                                    # "" / "none" / None -> None
    return None if v is None or str(v).lower() in ("", "none") else str(v)

# Probes trained in 3.1 (one per organism). They're tiny; ship them in the repo.
PROBE_DIRS = ("submission", "tutorials", "baselines")   # searched recursively — probes may live in baselines/<method>/ subfolders
PROBE_FILES = sorted({p for d in PROBE_DIRS for p in glob.glob(f"{d}/**/mean_probe_*.npz", recursive=True)})
assert PROBE_FILES, f"no mean_probe_*.npz found in {PROBE_DIRS}"

def pick_probe(ds_model, ds_lora):
    """The probe matching (model, lora); fall back to this model's base probe (no LoRA)."""
    want = _norm(ds_lora)
    base = None
    for f in PROBE_FILES:
        z = np.load(f, allow_pickle=True)
        if str(z["model_id"]) != ds_model:
            continue
        plora = _norm(z["lora_id"]) if "lora_id" in z.files else None
        if plora == want:
            return f, z                          # exact organism match
        if plora is None:
            base = (f, z)                         # remember base-model fallback
    if base is not None:
        print(f"[warn] no probe for lora={ds_lora!r}; falling back to base-model probe")
        return base
    raise FileNotFoundError(f"no probe for model={ds_model!r} lora={ds_lora!r} among {PROBE_FILES}")

PROBE_FILE, z = pick_probe(DS_MODEL, DS_LORA)
DIRECTION = z["direction"].astype(np.float32)              # (hidden,)
LAYER     = int(z["layer"])
BIAS      = float(z["bias"])      if "bias" in z.files      else 0.0
SCORE_STD = float(z["score_std"]) if "score_std" in z.files else 1.0
print(f"dataset organism: model={DS_MODEL} | lora={DS_LORA}")
print(f"using {PROBE_FILE} | layer={LAYER} | dim={DIRECTION.shape[0]} | "
      f"bias={BIAS:.4g} | score_std={SCORE_STD:.4g}")

## 2. Method
(same as tutorials/3.1 mean probe)

In [ ]:
import torch
from util import build_model, decoder_layers, run_full_session, load_examples, write_submission


def probe_preprocess(messages, tokenizer, max_len):
    """Match 3.1's `tokenize_eval`: detect span = final message content tokens, then
    left-trim to max_len. Returns (token_ids, (start, end)). Self-contained (cloudpickled)."""
    msgs = [{"role": m["role"], "content": m["content"]} for m in messages]
    # If the chat template rejects a `system` role (gemma), fold it into the first user turn.
    try:
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    except Exception:
        norm, pending = [], None
        for m in msgs:
            if m["role"] == "system" and not norm:
                pending = m["content"]; continue
            if not norm and m["role"] == "user" and pending is not None:
                norm.append({"role": "user", "content": pending + "\n\n" + m["content"]}); pending = None; continue
            norm.append(m)
        if pending is not None:
            norm = [{"role": "user", "content": pending}] + norm
        msgs = norm

    content = msgs[-1]["content"].strip()
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    enc = tokenizer(text, add_special_tokens=False)
    ids = enc["input_ids"]
    c = text.rfind(content) if content else -1
    if c == -1:
        s, e = 0, len(ids)
    else:
        s = enc.char_to_token(c)
        s = 0 if s is None else s
        e = enc.char_to_token(c + len(content))
        e = len(ids) if e is None else e
    if e <= s:                                    # empty span -> pool whole sequence
        s, e = 0, len(ids)
    if max_len and len(ids) > max_len:            # left-trim, preserving the response
        cut = len(ids) - max_len
        ids, s, e = ids[cut:], max(0, s - cut), max(0, e - cut)
        if e <= s:
            s, e = 0, len(ids)
    return ids, (s, e)


def detect(model, model_id, lora_id, batch, *, direction, bias, score_std, layer):
    """(B,) deception probabilities in [0,1], computed remotely at one layer."""
    with model.trace({"input_ids": batch.input_ids,
                      "attention_mask": batch.attention_mask}) as tracer:
        h = decoder_layers(model)[layer].output            # (B, seq, hidden)
        feats = batch.pool_response(h)                      # (B, hidden), mean over response
        dev = feats.device
        s = (feats.float() @ direction.to(dev) - bias) / score_std
        scores = torch.sigmoid(s).save()
        tracer.stop()                                       # skip layers > `layer` + LM head
    return scores

## 3. Run one session and write `submission.csv`

In [ ]:
BATCH_SIZE = 4 if "120B" in DS_MODEL else 8 
MAX_LEN = 256

scores = run_full_session(
    DATASET_NAME, detect,
    preprocess=probe_preprocess,
    batch_size=BATCH_SIZE,
    max_len=MAX_LEN,
    remote=True,
    limit=LIMIT,
    direction=torch.tensor(DIRECTION), bias=BIAS, score_std=SCORE_STD, layer=LAYER,
)
print(f"scored {len(scores)} rows in one session")

# Some dev datasets repeat source ids in their `index` column (it's not a unique key), and the
# grader rejects a submission with duplicate `index`. Collapse duplicates by averaging their
# scores -> one row per index. (On a dataset with unique indices this is a no-op.)
import numpy as np, pandas as pd
examples = load_examples(DATASET_NAME)
idx = list(examples["index"])[:len(scores)]
agg = (pd.DataFrame({"index": idx, "score": np.clip(np.asarray(scores, dtype=float), 0.0, 1.0)})
         .groupby("index", as_index=False)["score"].mean())
print(f"{len(idx)} rows -> {len(agg)} unique indices")
write_submission(agg["index"].values, agg["score"].values)